# **BASELINE MODELS 58 FEATURES**

In [5]:
!pip install catboost
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import (classification_report, accuracy_score, recall_score,
                             f1_score, precision_score, roc_auc_score, confusion_matrix,
                             roc_curve, auc, precision_recall_curve, ConfusionMatrixDisplay,
                             average_precision_score)
from sklearn.utils import resample

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier

warnings.filterwarnings('ignore')

In [6]:
data = pd.read_excel('/content/dataset.xlsx')

print("Shape inițial: ", data.shape)
display(data.head())

X = data.drop(columns=['Unnamed: 0', 'row', 'label'], axis=1)
y = data['label'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42,stratify=y)

print("\nDistributia datelor:")
print(f"Train set: {X_train.shape[0]} paciente")
print(f"Test set:  {X_test.shape[0]} paciente")
print(f"Prevalenta boala (Train): {y_train.mean()*100:.1f}%")

Shape inițial:  (886, 61)


,Unnamed: 0,Heavy / Extreme menstrual bleeding,Menstrual pain (Dysmenorrhea),Painful / Burning pain during sex (Dyspareunia),Pelvic pain,Irregular / Missed periods,Cramping,Abdominal pain / pressure,Back pain,Painful bowel movements,...,Hormonal problems,Bloating,Feeling sick,Decreased energy / Exhaustion,Abdominal Cramps during Intercourse,Insomnia / Sleeplessness,Acne / pimples,Loss of appetite,row,label
0,Heavy / Extreme menstrual bleeding;Fatigue / C...,1,1,1,1,1,1,1,1,1,...,0,1,1,1,1,0,0,0,0,1
1,Heavy / Extreme menstrual bleeding;Nausea;Pain...,1,1,1,1,1,1,1,1,1,...,0,0,1,1,1,0,0,1,1,1
2,Fatigue / Chronic fatigue;Nausea;Bloating;Back...,0,1,0,1,1,1,0,1,0,...,0,1,1,1,0,1,0,0,2,1
3,Heavy / Extreme menstrual bleeding;Fatigue / C...,1,0,0,0,0,1,1,1,0,...,0,1,0,1,0,0,0,0,3,1
4,Fatigue / Chronic fatigue;Painful / Burning pa...,0,1,1,0,0,1,0,1,1,...,0,0,0,0,0,0,0,0,4,1



Distributia datelor:
Train set: 708 paciente
Test set:  178 paciente
Prevalenta boala (Train): 53.5%


In [7]:
def evaluate_model(y_true, y_pred, y_proba, model_name="Model"):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0  #Recall
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision   = tp / (tp + fp) if (tp + fp) > 0 else 0

    acc   = accuracy_score(y_true, y_pred)
    f1    = f1_score(y_true, y_pred, zero_division=0)
    auc_roc = roc_auc_score(y_true, y_proba)
    auprc   = average_precision_score(y_true, y_proba)

    print(f"========== PERFORMANȚĂ: {model_name} ==========")
    print(f"AUC-ROC:     {auc_roc:.4f}")
    print(f"AUPRC:       {auprc:.4f}")
    print(f"Accuracy:    {acc:.4f}")
    print(f"F1-Score:    {f1:.4f}")
    print(f"Sensitivity: {sensitivity:.4f} (Recall / True Positive Rate)")
    print(f"Specificity: {specificity:.4f} (True Negative Rate)")
    print(f"Precision:   {precision:.4f} (Positive Predictive Value)")
    print("-------------------------------------------------")
    print("Confusion Matrix:")
    print(f"[{tn}] TN   [{fp}] FP")
    print(f"[{fn}] FN   [{tp}] TP\n")

    return {'auc': auc_roc, 'auprc': auprc, 'acc': acc, 'f1': f1,
            'sens': sensitivity, 'spec': specificity, 'prec': precision}

 **SVM Linear**

In [8]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: SVM (LINEAR KERNEL)")
best_svm_linear = SVC(
    kernel='linear',
    class_weight='balanced',
    probability=True,
    random_state=42
)
best_svm_linear.fit(X_train, y_train)

cv_outer_linear = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_linear = cross_val_score(
    best_svm_linear,
    X_train,
    y_train,
    cv=cv_outer_linear,
    scoring='roc_auc',
    n_jobs=-1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_linear.mean():.4f} ± {cv_scores_linear.std():.4f}\n")
y_pred_svm_linear  = best_svm_linear.predict(X_test)
y_proba_svm_linear = best_svm_linear.predict_proba(X_test)[:, 1]

svm_linear_test_metrics = evaluate_model(
    y_test,
    y_pred_svm_linear,
    y_proba_svm_linear,
    model_name="SVM (Linear)"
)

ANTRENARE MODEL: SVM (LINEAR KERNEL)
10-Fold CV (Train) AUC-ROC: 0.9779 ± 0.0156

========== PERFORMANȚĂ: SVM (Linear) ==========
AUC-ROC:     0.9781
AUPRC:       0.9827
Accuracy:    0.8933
F1-Score:    0.9005
Sensitivity: 0.9053 (Recall / True Positive Rate)
Specificity: 0.8795 (True Negative Rate)
Precision:   0.8958 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[73] TN   [10] FP
[9] FN   [86] TP



**SVM RBF**

In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: SVM (RBF KERNEL)")
best_svm_rbf = SVC(
    kernel='rbf',
    class_weight='balanced',
    probability=True,
    random_state=42
)
best_svm_rbf.fit(X_train, y_train)

cv_outer_rbf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_rbf = cross_val_score(
    best_svm_rbf,
    X_train,
    y_train,
    cv=cv_outer_rbf,
    scoring='roc_auc',
    n_jobs=-1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_rbf.mean():.4f} ± {cv_scores_rbf.std():.4f}\n")
y_pred_svm_rbf  = best_svm_rbf.predict(X_test)
y_proba_svm_rbf = best_svm_rbf.predict_proba(X_test)[:, 1]

svm_rbf_test_metrics = evaluate_model(
    y_test,
    y_pred_svm_rbf,
    y_proba_svm_rbf,
    model_name="SVM (RBF)"
)

ANTRENARE MODEL: SVM (RBF KERNEL)
10-Fold CV (Train) AUC-ROC: 0.9807 ± 0.0135

========== PERFORMANȚĂ: SVM (RBF) ==========
AUC-ROC:     0.9829
AUPRC:       0.9859
Accuracy:    0.9157
F1-Score:    0.9180
Sensitivity: 0.8842 (Recall / True Positive Rate)
Specificity: 0.9518 (True Negative Rate)
Precision:   0.9545 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[79] TN   [4] FP
[11] FN   [84] TP



**RANDOM FOREST**

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: RANDOM FOREST")

best_rf = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
best_rf.fit(X_train, y_train)
cv_outer_rf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_rf = cross_val_score(
    best_rf,
    X_train,
    y_train,
    cv=cv_outer_rf,
    scoring='roc_auc',
    n_jobs=-1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}\n")
y_pred_rf  = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:, 1]

rf_test_metrics = evaluate_model(
    y_test,
    y_pred_rf,
    y_proba_rf,
    model_name="Random Forest"
)

ANTRENARE MODEL: RANDOM FOREST
10-Fold CV (Train) AUC-ROC: 0.9760 ± 0.0100

========== PERFORMANȚĂ: Random Forest ==========
AUC-ROC:     0.9819
AUPRC:       0.9858
Accuracy:    0.9438
F1-Score:    0.9468
Sensitivity: 0.9368 (Recall / True Positive Rate)
Specificity: 0.9518 (True Negative Rate)
Precision:   0.9570 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[79] TN   [4] FP
[6] FN   [89] TP



**XGBoost**

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
import numpy as np

print("ANTRENARE MODEL: XGBOOST")

ratio = float(np.sum(y_train == 0)) / np.sum(y_train == 1)

best_xgb = XGBClassifier(
    scale_pos_weight=ratio,
    random_state=42,
    n_jobs=-1
)

best_xgb.fit(X_train, y_train)

cv_outer_xgb = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_xgb = cross_val_score(
    best_xgb,
    X_train,
    y_train,
    cv=cv_outer_xgb,
    scoring='roc_auc',
    n_jobs=-1
)

print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}\n")

y_pred_xgb  = best_xgb.predict(X_test)
y_proba_xgb = best_xgb.predict_proba(X_test)[:, 1]

xgb_test_metrics = evaluate_model(
    y_test,
    y_pred_xgb,
    y_proba_xgb,
    model_name="XGBoost"
)

ANTRENARE MODEL: XGBOOST
10-Fold CV (Train) AUC-ROC: 0.9769 ± 0.0107

========== PERFORMANȚĂ: XGBoost ==========
AUC-ROC:     0.9815
AUPRC:       0.9848
Accuracy:    0.9045
F1-Score:    0.9128
Sensitivity: 0.9368 (Recall / True Positive Rate)
Specificity: 0.8675 (True Negative Rate)
Precision:   0.8900 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[72] TN   [11] FP
[6] FN   [89] TP



**CATBoost**

In [ ]:
!pip install catboost

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

print("ANTRENARE MODEL: CATBOOST")

best_catboost = CatBoostClassifier(
    scale_pos_weight=float(ratio),
    random_seed=42,
    verbose=False,
    allow_writing_files=False
)

best_catboost.fit(X_train, y_train)
cv_outer_cat = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
cv_scores_cat = cross_val_score(
    best_catboost,
    X_train,
    y_train,
    cv=cv_outer_cat,
    scoring='roc_auc',
    n_jobs=-1
)
print(f"10-Fold CV (Train) AUC-ROC: {cv_scores_cat.mean():.4f} ± {cv_scores_cat.std():.4f}\n")
y_pred_cat  = best_catboost.predict(X_test)
y_proba_cat = best_catboost.predict_proba(X_test)[:, 1]

catboost_test_metrics = evaluate_model(
    y_test,
    y_pred_cat,
    y_proba_cat,
    model_name="CatBoost"
)

ANTRENARE MODEL: CATBOOST
10-Fold CV (Train) AUC-ROC: 0.9812 ± 0.0110

========== PERFORMANȚĂ: CatBoost ==========
AUC-ROC:     0.9845
AUPRC:       0.9875
Accuracy:    0.9213
F1-Score:    0.9271
Sensitivity: 0.9368 (Recall / True Positive Rate)
Specificity: 0.9036 (True Negative Rate)
Precision:   0.9175 (Positive Predictive Value)
-------------------------------------------------
Confusion Matrix:
[75] TN   [8] FP
[6] FN   [89] TP

